In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl
import prusti_analysis as pa

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(300)
pl.Config.set_tbl_rows(80)

In [ ]:
db_paths = sorted(Path(".").glob("prusti-*.db"))
print(f"Found {len(db_paths)} databases:")
for p in db_paths:
    print(f"  {p.name}")

df = pa.transform(pa.load_dbs(db_paths))
print(f"\n{len(df)} total rows")

In [ ]:
# Overall success / fail / timeout counts per version
print(df.group_by(["db", "success"]).agg(pl.len().alias("count"))
        .sort(["db", "success"]))

In [ ]:
# Failure count per category per version — pivot so versions are columns
fails = df.filter(pl.col("success") == "fail")

counts = (
    fails
    .group_by(["db", "category"])
    .agg(pl.len().alias("count"))
)

dbs = sorted(df["db"].unique().to_list())
pivot = counts.pivot(on="db", index="category", values="count").fill_null(0)

# Sort by total count across all versions
pivot = (
    pivot
    .with_columns(pl.sum_horizontal([c for c in dbs if c in pivot.columns]).alias("_total"))
    .sort("_total", descending=True)
    .drop("_total")
)

print(pivot)

In [ ]:
# Delta between first and last version for each category
if len(dbs) >= 2:
    first, last = dbs[0], dbs[-1]
    delta = (
        pivot
        .with_columns(
            (pl.col(last) - pl.col(first)).alias("delta")
        )
        .select(["category", first, last, "delta"])
        .sort("delta")
    )
    print(f"Delta from {first} → {last}")
    print(delta)
else:
    print("Need at least 2 databases to compute deltas.")